# Falcao — National Labor-Justice Jurisprudence (CSJT)

`FalcaoScraper` queries the **Jurisprudencia Nacional** system of the Brazilian
Labor Justice (Justica do Trabalho), hosted by the CSJT at
<https://jurisprudencia.jt.jus.br>. A single search endpoint spans the **TST**
and all **24 TRTs** across five document collections.

| Method | Description |
|--------|-------------|
| `cjsg` | Jurisprudence search — returns a `DataFrame` |
| `cjsg_download` | Downloads the raw JSON pages to disk |
| `cjsg_parse` | Parses a folder produced by `cjsg_download` |

**Collections** (`colecao` argument): `acordaos` (default), `sentencas`,
`decisoesmonocraticas`, `precedentes`, `recursorevista`.

> Notes: the public (unauthenticated) endpoint only allows page sizes of **5**
> or **10** (`tamanho_pagina`) and caps results at 10,000 per query.

## Basic search

In [ ]:
import juscraper as jus

falcao = jus.scraper("falcao")
df = falcao.cjsg("dano moral", paginas=1)
print(df.shape)
df.head(3)

## Available columns

In [ ]:
df.columns.tolist()

## Preview an ementa

The `acordaos` collection ships the full ementa (HTML).

In [ ]:
print(df["ementa"].iloc[0][:300])

## Using filters

Filters are validated by the `InputCJSGFalcao` pydantic schema. Multi-value
filters accept a string or a list of strings.

In [ ]:
df_filtrado = falcao.cjsg(
    "hora extra",
    paginas=1,
    tribunais=["TST", "TRT3"],       # court(s)
    ordenacao="mais_recente",         # mais_relevante | mais_recente | menos_recente
    data_juntada_inicio="2023-01-01",  # ISO or DD/MM/YYYY, filters dataJuntada
    data_juntada_fim="2023-12-31",
)
df_filtrado[["processo", "tribunal", "data_juntada"]].head()

## Querying different collections

The same search works across collections; each returns its own document shape,
normalized to a common core (`processo`, `colecao`, `tribunal`, ...).

In [ ]:
df_sent = falcao.cjsg("acidente de trabalho", paginas=1, colecao="sentencas")
print("sentencas:", df_sent.shape)

df_sum = falcao.cjsg("dano moral", paginas=1, colecao="precedentes")
print("precedentes:", df_sum.shape)
df_sum[["processo", "colecao", "tribunal"]].head()

## Download and parse separately

`cjsg_download` saves one raw JSON file per page (named
`{colecao}_{page}.json`); `cjsg_parse` reads a folder back into a `DataFrame`.

In [ ]:
import tempfile, os, json

with tempfile.TemporaryDirectory() as tmp:
    pasta = falcao.cjsg_download("dano moral", paginas=1, diretorio=tmp)
    arquivos = os.listdir(pasta)
    print("files:", arquivos)

    raw = json.load(open(os.path.join(pasta, arquivos[0])))
    print("quantidadeTotal:", raw["quantidadeTotal"])
    print("raw keys:", list(raw.keys()))

    df_parsed = falcao.cjsg_parse(pasta)
    print("parsed:", df_parsed.shape)